# 08: Qwen3-8B 비추론 비증강 QLoRA 학습과 평가, Unsloth 데스크톱

07의 실제 Unsloth 학습과 같은 코드 및 파라미터를 사용하고, `HYPER_PARAMS["augmentation"]["probability"]`만 0.5에서 0.0으로 바꿉니다. 문장 결합을 적용하지 않고 원본 `train.jsonl`을 읽습니다. 비추론 프롬프트를 제외한 JSON 응답과 EOS에만 손실을 적용합니다.

커널 작업 디렉터리를 `experiments/08`로 설정하고 프로젝트 `.venv/Scripts/python.exe`를 선택합니다. `prefix_path='./data/'`를 기준으로 08 내부에 어댑터, 체크포인트, 실행 설정과 학습 이력을 저장합니다. 07의 어댑터를 이어 학습하지 않으며 같은 revision의 기본 모델에서 시작합니다. 전체 셀을 순서대로 실행하면 학습 종료 후 하단의 평가가 자동으로 이어집니다. 학습 후 같은 커널에서 평가 변수와 함수를 재정의하고 평가합니다.

## 의존성과 고정 조건

07과 같은 기존 가상환경과 루트 `pyproject.toml`의 `unsloth` 선택 의존성을 사용합니다. 필요 시 커널을 종료하고 프로젝트 루트에서 다음 명령으로 준비합니다.

```powershell
.\.venv\Scripts\python.exe -m pip install --extra-index-url https://download.pytorch.org/whl/cu128 ".[unsloth]"
.\.venv\Scripts\python.exe -m pip check
```

비교 실행은 07과 같은 RTX 4070, BF16, Unsloth 2026.10.9와 transformers.Trainer를 사용합니다. 모델 revision, NF4 4비트 및 이중 양자화, 3에포크, 배치 1, 누적 8, 학습률 0.0002, 시드 42, 최대 1,280토큰, LoRA rank 16, alpha 32, dropout 0.05와 Unsloth 체크포인팅을 유지합니다. 실제 환경은 실행 JSON에 기록합니다.

증강 제거에 따른 토큰 수 및 작업 노출 감소를 보상하기 위해 에포크나 스텝을 늘리지 않습니다. 완전 일치율과 값 토큰 신뢰도는 하단 평가 영역에서 계산합니다. 실험 설계와 비교 범위는 [실험 안내](README.md)를 따릅니다.


## 1. 파라미터와 환경


In [1]:
RUNTIME_PARAMS = {'prefix_path': './data/',
 'model_name': 'Qwen/Qwen3-8B',
 'train_file': 'train.jsonl',
 'test_file': 'test.jsonl',
 'output_dir': 'models/checkpoints',
 'adapter_path': 'models',
 'run_config_file': 'result/run_config.json',
 'training_history_file': 'result/training_history.json',
 'report_file': 'result/evaluation.json',
 'device': 'cuda',
 'model_revision': 'b968826d9c46dd6066d109eabc6255188de91218'}

HYPER_PARAMS = {
    # 07과 같은 구현에서 확률 0.0으로 모든 문장 결합 증강을 끕니다.
    'augmentation': {
        'probability': 0.0,
        'max_concat': 2,
        'delimiters': [' ', '  ', ' / ', ' . ', '\n', ' , ', ' + '],
    },
    'chat_template': {'enable_thinking': False},
    'training': {
        'epochs': 3,
        'batch_size': 1,
        'grad_accum': 8,
        'learning_rate': 0.0002,
        'max_seq_length': 1280,
        'seed': 42,
    },
    'lora': {
        'r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05,
        'task_type': 'CAUSAL_LM', 'bias': 'none',
        'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                           'gate_proj', 'up_proj', 'down_proj'],
    },
    'quantization': {
        'load_in_4bit': True,
        'bnb_4bit_quant_type': 'nf4',
        'bnb_4bit_use_double_quant': True,
    },
    # auto는 BF16 지원 GPU에서 BF16, 그 외에는 FP16을 선택합니다.
    'precision': {'compute_dtype': 'auto', 'frozen_embedding_dtype': 'auto'},
    'model_loading': {
        'device_map': {'': 0}, 'attn_implementation': 'sdpa', 'low_cpu_mem_usage': True,
        'use_exact_model_name': True, 'fix_tokenizer': False, 'offload_embedding': False,
    },
    'tokenizer': {'padding_side': 'right'},
    'kbit_training': {
        # Unsloth의 메모리 절약 체크포인팅을 사용합니다.
        'use_gradient_checkpointing': 'unsloth',
    },
    'trainer': {
        'per_device_eval_batch_size': 1,
        'lr_scheduler_type': 'cosine', 'warmup_steps': 10,
        'logging_steps': 10, 'save_strategy': 'epoch', 'eval_strategy': 'epoch',
        'save_total_limit': 2, 'optim': 'paged_adamw_8bit',
        'dataloader_num_workers': 0, 'dataloader_pin_memory': True,
        'report_to': 'none', 'remove_unused_columns': False,
        'prediction_loss_only': True, 'label_names': ['labels'],
    },
    'collator': {'pad_to_multiple_of': 8, 'return_tensors': 'pt', 'padding': True},
}


## 2. 라이브러리


In [2]:
import hashlib
import json
import os
import sys
import tempfile
import time
import platform
import random
from copy import deepcopy
from pathlib import Path

# 모델 저장 위치와 분리된 시스템 임시 경로에 컴파일 캐시를 둡니다.
os.environ.setdefault(
    "UNSLOTH_COMPILE_LOCATION",
    str(Path(tempfile.gettempdir()) / "siteboard_unsloth_compiled_cache"),
)
if "unsloth" not in sys.modules and any(
    name in sys.modules for name in ("transformers", "peft", "trl")
):
    raise RuntimeError("커널을 재시작한 뒤 이 노트북을 위에서부터 실행하세요. Unsloth를 먼저 가져와야 합니다.")

from unsloth import FastLanguageModel
import torch
from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
)


[fla.utils._device|WARNING]Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1028 09:28:07.232000 5224 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!


## 3. 경로와 결과 저장


In [3]:
def runtime_path(field):
    return str(Path(RUNTIME_PARAMS["prefix_path"]) / RUNTIME_PARAMS[field])


def write_json(path, value, mode="x"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open(mode, encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2)
        stream.write("\n")


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_dtype(name):
    if name == "auto":
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return getattr(torch, name)


## 4. 데이터 읽기


In [4]:
def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as stream:
        records = [json.loads(line) for line in stream if line.strip()]
    if not records:
        raise ValueError(f"입력 JSONL이 비어 있습니다: {path}")
    return records


def prepare_training_data():
    train_records = read_jsonl(runtime_path("train_file"))
    valid_records = read_jsonl(runtime_path("test_file"))
    return train_records, valid_records, {
        "train_count": len(train_records), "test_count": len(valid_records),
    }


## 5. 원본 입력과 비추론 토큰화


In [5]:
class QwenChatDataset:
    """같은 현장의 레코드를 랜덤 결합하고 JSON 응답 구간만 학습합니다."""

    def __init__(self, records, tokenizer, max_seq_length, is_train,
                 aug_prob, aug_max_concat, delimiters):
        self.records = records
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length
        self.is_train = is_train
        self.aug_prob = aug_prob
        self.aug_max_concat = aug_max_concat
        self.delimiters = delimiters
        self.site_to_indices = {}
        self.components = []
        for index, record in enumerate(records):
            self.site_to_indices.setdefault(record["metadata"]["site"], []).append(index)
            messages = record["messages"]
            rules, text = messages[1]["content"].split("[보드판 텍스트]\n", 1)
            self.components.append((messages[0]["content"], rules, text,
                                    json.loads(messages[2]["content"])))

    def __len__(self):
        return len(self.records)

    def encode_messages(self, messages):
        # 비추론 프롬프트와 JSON 응답을 분리해 경계 토큰까지 정확히 마스킹합니다.
        prompt = self.tokenizer.apply_chat_template(
            messages[:2], tokenize=False, add_generation_prompt=True,
            **HYPER_PARAMS["chat_template"],
        )
        prompt_ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        response_ids = self.tokenizer.encode(messages[2]["content"], add_special_tokens=False)
        response_ids.append(self.tokenizer.eos_token_id)
        input_ids = prompt_ids + response_ids
        if len(input_ids) > self.max_seq_length:
            raise ValueError(f"학습 입력과 정답 {len(input_ids)}토큰이 상한 {self.max_seq_length}을 넘습니다. 잘라서 학습하지 않습니다.")
        return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids),
                "labels": [-100] * len(prompt_ids) + response_ids}

    def __getitem__(self, index):
        messages = self.records[index]["messages"]
        if self.is_train and random.random() < self.aug_prob:
            system, rules, text, answers = self.components[index]
            texts, combined = [text], list(answers)
            pool = self.site_to_indices[self.records[index]["metadata"]["site"]]
            for _ in range(random.randint(2, self.aug_max_concat) - 1):
                # 06과 같이 자기 자신도 포함한 동일 현장 풀에서 복원 추출합니다.
                partner = random.choice(pool)
                _, _, partner_text, partner_answers = self.components[partner]
                texts.append(partner_text)
                combined.extend(partner_answers)
            messages = [
                {"role": "system", "content": system},
                {"role": "user", "content": rules + "[보드판 텍스트]\n" + random.choice(self.delimiters).join(texts)},
                {"role": "assistant", "content": json.dumps(combined, ensure_ascii=False, indent=2)},
            ]
        return self.encode_messages(messages)


def build_dataset(records, tokenizer, is_train):
    settings = HYPER_PARAMS["augmentation"]
    return QwenChatDataset(
        records, tokenizer, HYPER_PARAMS["training"]["max_seq_length"], is_train,
        settings["probability"], settings["max_concat"], settings["delimiters"],
    )




## 6. 모델과 학습기


In [6]:
def load_tokenizer():
    # 기존 마스킹과 평가에 사용하는 공식 템플릿을 그대로 읽습니다.
    tokenizer = AutoTokenizer.from_pretrained(
        RUNTIME_PARAMS["model_name"], revision=RUNTIME_PARAMS["model_revision"],
        **HYPER_PARAMS["tokenizer"],
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_qlora_model():
    dtype = resolve_dtype(HYPER_PARAMS["precision"]["compute_dtype"])
    # 반환된 토크나이저 대신 위에서 읽은 공식 토크나이저로 학습하고 저장합니다.
    model, _ = FastLanguageModel.from_pretrained(
        model_name=RUNTIME_PARAMS["model_name"], revision=RUNTIME_PARAMS["model_revision"],
        max_seq_length=HYPER_PARAMS["training"]["max_seq_length"], dtype=dtype,
        load_in_4bit=HYPER_PARAMS["quantization"]["load_in_4bit"],
        quantization_config=BitsAndBytesConfig(
            bnb_4bit_compute_dtype=dtype, **HYPER_PARAMS["quantization"],
        ),
        use_gradient_checkpointing=HYPER_PARAMS["kbit_training"]["use_gradient_checkpointing"],
        random_state=HYPER_PARAMS["training"]["seed"],
        **HYPER_PARAMS["model_loading"],
    )
    actual_revision = getattr(model.config, "_commit_hash", None)
    if actual_revision is not None and actual_revision != RUNTIME_PARAMS["model_revision"]:
        raise ValueError(f"요청한 기본 모델 revision과 다릅니다: {actual_revision}")
    model.config.use_cache = False
    return model


def add_lora_adapter(model):
    settings = dict(HYPER_PARAMS["lora"])
    if settings.pop("task_type") != "CAUSAL_LM":
        raise ValueError("이 노트북은 CAUSAL_LM LoRA 학습만 지원합니다.")
    # Unsloth API가 k비트 준비와 LoRA 패치를 함께 수행합니다.
    model = FastLanguageModel.get_peft_model(
        model,
        random_state=HYPER_PARAMS["training"]["seed"],
        **settings, **HYPER_PARAMS["kbit_training"],
    )
    dtype = resolve_dtype(HYPER_PARAMS["precision"]["frozen_embedding_dtype"])
    for module in (model.get_input_embeddings(), model.get_output_embeddings()):
        if module is not None and not module.weight.requires_grad:
            module.to(dtype=dtype)
    for config in model.peft_config.values():
        config.base_model_name_or_path = RUNTIME_PARAMS["model_name"]
        config.revision = RUNTIME_PARAMS["model_revision"]
    model.config.use_cache = False
    model.print_trainable_parameters()
    return model


def build_trainer(model, tokenizer, train_records, valid_records):
    settings = HYPER_PARAMS["training"]
    train_dataset = build_dataset(train_records, tokenizer, is_train=True)
    valid_dataset = build_dataset(valid_records, tokenizer, is_train=False)
    dtype = resolve_dtype(HYPER_PARAMS["precision"]["compute_dtype"])
    args = TrainingArguments(
        output_dir=runtime_path("output_dir"), num_train_epochs=settings["epochs"],
        per_device_train_batch_size=settings["batch_size"],
        gradient_accumulation_steps=settings["grad_accum"],
        learning_rate=settings["learning_rate"], seed=settings["seed"],
        bf16=dtype == torch.bfloat16, fp16=dtype == torch.float16,
        **HYPER_PARAMS["trainer"],
    )
    # Dataset을 직접 전달해 매 접근 시점의 랜덤 결합과 labels를 유지합니다.
    trainer = Trainer(
        model=model, args=args, processing_class=tokenizer,
        train_dataset=train_dataset, eval_dataset=valid_dataset,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, **HYPER_PARAMS["collator"]),
    )
    return trainer


## 7. 실행 정보와 학습


In [7]:
def collect_execution_metadata(model=None):
    from importlib.metadata import PackageNotFoundError, version

    packages = {}
    for name in ("torch", "transformers", "peft", "accelerate", "bitsandbytes",
                 "unsloth", "unsloth_zoo", "trl", "triton", "triton-windows",
                 "torchvision", "xformers"):
        try:
            packages[name] = version(name)
        except PackageNotFoundError:
            packages[name] = None
    return {
        "python": platform.python_version(), "python_executable": sys.executable,
        "platform": platform.platform(), "packages": packages,
        "training_backend": "unsloth", "trainer": "transformers.Trainer",
        "gradient_checkpointing": HYPER_PARAMS["kbit_training"]["use_gradient_checkpointing"],
        "unsloth_compile_location": os.environ.get("UNSLOTH_COMPILE_LOCATION"),
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "cuda": torch.version.cuda,
        "base_model": RUNTIME_PARAMS["model_name"],
        "base_model_revision": getattr(getattr(model, "config", None), "_commit_hash", None)
            or RUNTIME_PARAMS["model_revision"],
        "compute_dtype": str(resolve_dtype(HYPER_PARAMS["precision"]["compute_dtype"])),
        "input_embedding_dtype": str(model.get_input_embeddings().weight.dtype) if model is not None else None,
        "output_embedding_dtype": str(model.get_output_embeddings().weight.dtype) if model is not None else None,
        "gpu_total_bytes": torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
        "peak_allocated_bytes": torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None,
        "peak_reserved_bytes": torch.cuda.max_memory_reserved() if torch.cuda.is_available() else None,
    }


def check_training_environment():
    if RUNTIME_PARAMS["device"] != "cuda" or not torch.cuda.is_available():
        raise RuntimeError("이 노트북은 CUDA GPU 런타임이 필요합니다.")
    dtype = resolve_dtype(HYPER_PARAMS["precision"]["compute_dtype"])
    if dtype not in (torch.bfloat16, torch.float16):
        raise ValueError("compute_dtype은 auto, bfloat16 또는 float16을 사용하세요.")
    if dtype == torch.bfloat16 and not torch.cuda.is_bf16_supported():
        raise ValueError("이 GPU는 BF16을 지원하지 않습니다. compute_dtype을 float16으로 지정하세요.")
    for key in ("train_file", "test_file"):
        if not Path(runtime_path(key)).is_file():
            raise FileNotFoundError(f"입력 JSONL이 없습니다. prefix_path를 확인하세요: {runtime_path(key)}")


def check_output_paths():
    for key in ("run_config_file", "training_history_file", "report_file"):
        if Path(runtime_path(key)).exists():
            raise FileExistsError(f"기존 산출물이 있습니다: {runtime_path(key)}")
    adapter = Path(runtime_path("adapter_path"))
    if adapter.exists() and any(p.name != "README.md" for p in adapter.iterdir()):
        raise FileExistsError(f"기존 모델 또는 체크포인트가 있습니다: {adapter}")
    output = Path(runtime_path("output_dir"))
    if output.exists() and any(output.iterdir()):
        raise FileExistsError(f"기존 체크포인트가 있습니다: {output}")


def run_training():
    check_output_paths()
    check_training_environment()
    set_seed(HYPER_PARAMS["training"]["seed"])
    train_records, valid_records, prepared_data = prepare_training_data()
    tokenizer = load_tokenizer()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    model = add_lora_adapter(load_qlora_model())
    trainer = build_trainer(model, tokenizer, train_records, valid_records)
    paths = [runtime_path(key) for key in ("train_file", "test_file")]
    run_config = {
        "runtime_params": deepcopy(RUNTIME_PARAMS), "hyper_params": deepcopy(HYPER_PARAMS),
        "prepared_data": prepared_data, "prepared_file_hashes": {path: file_sha256(path) for path in paths},
        "execution": collect_execution_metadata(model), "status": "training_started",
    }
    write_json(runtime_path("run_config_file"), run_config)
    try:
        trainer.train()
        destination = Path(runtime_path("adapter_path"))
        model.save_pretrained(destination)
        tokenizer.save_pretrained(destination)
    except (Exception, KeyboardInterrupt) as error:
        run_config.update(
            status="training_interrupted" if isinstance(error, KeyboardInterrupt) else "training_failed",
            error={"type": type(error).__name__, "message": str(error)},
            duration_seconds=time.perf_counter() - started,
            execution=collect_execution_metadata(model),
        )
        write_json(runtime_path("run_config_file"), run_config, mode="w")
        write_json(runtime_path("training_history_file"), trainer.state.log_history)
        raise
    write_json(runtime_path("training_history_file"), trainer.state.log_history)
    run_config.update(status="training_completed", duration_seconds=time.perf_counter() - started,
                      execution=collect_execution_metadata(model))
    write_json(runtime_path("run_config_file"), run_config, mode="w")
    return str(destination)


## 8. 설정 확인 후 실행

기존 결과가 있으면 중단합니다. 길이 상한을 초과하는 실제 학습 샘플도 자르지 않고 중단합니다. `train.jsonl`은 학습에, `test.jsonl`은 증강 없는 loss 평가에 사용합니다.

| 출력 | 기존 학습과 같은 형태 |
| --- | --- |
| `data/models/` | 병합하지 않은 PEFT 어댑터와 공식 토크나이저 |
| `data/models/checkpoints/checkpoint-*/` | epoch별 Trainer 체크포인트, 최대 2개 유지 |
| `data/result/run_config.json` | runtime_params, hyper_params, prepared_data, 입력 해시, execution, status와 소요 시간 |
| `data/result/training_history.json` | Trainer log_history 배열, 학습 loss와 epoch별 eval_loss |

중단이나 오류가 발생하면 실행 상태와 수집된 학습 이력을 남기고 예외를 다시 발생시킵니다. 어댑터 병합과 완전 일치 평가는 수행하지 않습니다. 완료 후 아래 평가 셀을 이어 실행합니다. 학습 메모리를 정리한 뒤 같은 커널에서 평가 변수와 함수를 재정의하고 같은 `prefix_path`, 기본 모델 revision과 dtype으로 평가합니다.


In [8]:
print(json.dumps(RUNTIME_PARAMS, ensure_ascii=False, indent=2))
print(json.dumps(HYPER_PARAMS, ensure_ascii=False, indent=2))


{
  "prefix_path": "./data/",
  "model_name": "Qwen/Qwen3-8B",
  "train_file": "train.jsonl",
  "test_file": "test.jsonl",
  "output_dir": "models/checkpoints",
  "adapter_path": "models",
  "run_config_file": "result/run_config.json",
  "training_history_file": "result/training_history.json",
  "report_file": "result/evaluation.json",
  "device": "cuda",
  "model_revision": "b968826d9c46dd6066d109eabc6255188de91218"
}
{
  "augmentation": {
    "probability": 0.0,
    "max_concat": 2,
    "delimiters": [
      " ",
      "  ",
      " / ",
      " . ",
      "\n",
      " , ",
      " + "
    ]
  },
  "chat_template": {
    "enable_thinking": false
  },
  "training": {
    "epochs": 3,
    "batch_size": 1,
    "grad_accum": 8,
    "learning_rate": 0.0002,
    "max_seq_length": 1280,
    "seed": 42
  },
  "lora": {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "task_type": "CAUSAL_LM",
    "bias": "none",
    "target_modules": [
      "q_proj",
      "k_proj",
      "

In [ ]:
adapter_path = run_training()
print("어댑터와 토크나이저 저장:", adapter_path)


==((====))==  Unsloth 2026.10.9: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4070. Num GPUs = 1. Max memory: 11.994 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Unsloth 2026.10.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen3DecoderLayer).
trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,568 | Num Epochs = 3 | Total steps = 588
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss


# 학습 종료 후 평가

전체 셀을 순서대로 실행하면 같은 커널에서 학습에 이어 평가합니다. 학습 함수 종료 후 남은 순환 참조와 CUDA 캐시를 정리하고 아래에서 평가 파라미터, 라이브러리와 함수를 다시 정의합니다. 커널 재시작과 별도 프로세스는 사용하지 않습니다.

아래는 `evaluate_desktop.ipynb`의 평가 셀입니다. 모델 revision과 입력, 어댑터 및 결과 경로는 학습 설정을 이어받고 평가용 파라미터는 다시 정의합니다. 같은 커널이므로 학습 시 적용된 Unsloth 패치가 유지됩니다. 기존 평가 결과의 덮어쓰기 방지는 그대로 적용합니다.


In [10]:
import gc

TRAINING_RUNTIME_PARAMS = deepcopy(RUNTIME_PARAMS)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()


# 08: Qwen3-8B 비추론 비증강 QLoRA 평가, 데스크톱

07의 대응 노트북과 같은 모델, 프롬프트 및 파라미터를 사용하며 학습 증강 확률은 0.0입니다. `prefix_path`는 08의 데이터 디렉터리를 가리킵니다. 상단 딕셔너리를 확인한 뒤 순서대로 실행합니다. 다른 실험의 데이터와 어댑터를 읽지 않습니다.

07의 실제 실행과 비교할 때는 `train_desktop_unsloth.ipynb`로 학습하고 `evaluate_desktop.ipynb`로 평가합니다. 일반 학습 및 Colab 경로는 별도 실행 구성으로 보존하며 백엔드와 환경 차이를 증강 효과에 섞어 해석하지 않습니다. 자세한 실행 및 비교 조건은 [실험 안내](README.md)를 따릅니다.


## 1. 파라미터와 환경


In [11]:
RUNTIME_PARAMS = {'prefix_path': 'data',
 'model_name': 'Qwen/Qwen3-8B',
 'model_revision': 'b968826d9c46dd6066d109eabc6255188de91218',
 'adapter_path': 'models',
 'test_file': 'test.jsonl',
 'split_meta_file': 'split_meta.json',
 'run_config_file': 'result/run_config.json',
 'report_file': 'result/evaluation.json',
 'device': 'cuda'}

HYPER_PARAMS = {'chat_template': {'enable_thinking': False},
 'quantization': {'load_in_4bit': True,
                  'bnb_4bit_quant_type': 'nf4',
                  'bnb_4bit_use_double_quant': True},
 'precision': {'compute_dtype': 'auto', 'frozen_embedding_dtype': 'auto'},
 'model_loading': {'device_map': {'': 0}, 'attn_implementation': 'sdpa', 'low_cpu_mem_usage': True},
 'tokenizer': {'padding_side': 'left'},
 'evaluation': {'batch_size': 1,
                'max_new_tokens': 512,
                'do_sample': False}}

RUNTIME_PARAMS.update({
    key: TRAINING_RUNTIME_PARAMS[key]
    for key in ("prefix_path", "model_name", "model_revision", "adapter_path",
                "test_file", "run_config_file", "report_file", "device")
})


## 2. 라이브러리


In [12]:
import hashlib
import json
import math
import platform
import re
import sys
import time
from collections import Counter
from copy import deepcopy
from pathlib import Path

import torch
from peft import PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


## 3. 경로와 파일 해시


In [13]:
def runtime_path(field):
    return str(Path(RUNTIME_PARAMS["prefix_path"]) / RUNTIME_PARAMS[field])


def write_json(path, value, mode="x"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open(mode, encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2)
        stream.write("\n")


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_dtype(name):
    if name == "auto":
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return getattr(torch, name)


## 4. 평가 데이터 읽기


In [14]:
def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]



## 5. 엄격한 JSON 채점과 값 토큰 신뢰도


In [15]:
def unique_object(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"중복 JSON 키: {key}")
        result[key] = value
    return result


def parse_prediction(text):
    """필드와 값 타입을 검사하고, 유효하지 않은 출력은 None으로 반환합니다."""
    try:
        data = json.loads(text, object_pairs_hook=unique_object)
        if not isinstance(data, list):
            return None
        for item in data:
            if not isinstance(item, dict) or set(item) != {"workType", "spec", "quantity"}:
                return None
            if not isinstance(item["workType"], str) or not item["workType"].strip():
                return None
            if not isinstance(item["spec"], str):
                return None
            item["workType"].encode("utf-8")
            item["spec"].encode("utf-8")
            if type(item["quantity"]) not in (int, float) or not math.isfinite(item["quantity"]):
                return None
        return data
    except (ValueError, TypeError, OverflowError, RecursionError):
        return None


def parse_and_normalize(text):
    data = parse_prediction(text)
    if data is None:
        return None
    return [(item["workType"].strip(), item["spec"].strip(), round(item["quantity"], 4)) for item in data]


def build_token_spans(tokenizer, token_ids, probabilities, pred_text):
    """Qwen 바이트 토큰을 복원해 한글 분할 토큰에도 같은 값 구간을 연결합니다."""
    visible = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
    missing = [value for value in range(256) if value not in visible]
    byte_decoder = dict(zip(map(chr, visible + list(range(256, 256 + len(missing)))), visible + missing))
    spans = []
    restored = bytearray()
    try:
        for token, probability in zip(tokenizer.convert_ids_to_tokens(token_ids), probabilities):
            start = len(restored)
            restored.extend(byte_decoder[char] for char in token)
            spans.append((start, len(restored), probability))
    except KeyError:
        return None
    if len(spans) != len(token_ids) or bytes(restored) != pred_text.encode("utf-8"):
        return None
    return spans


def extract_values_with_confidence(pred_text, token_spans):
    """값과 겹치는 생성 토큰을 집계하며 대응이 미확인인 값은 None으로 둡니다."""
    data = parse_prediction(pred_text)
    if data is None:
        return [], None, None
    predictions = [dict(item) for item in data]
    for item in predictions:
        for field in ("workType", "spec", "quantity"):
            item[f"{field}_confidence"] = None
    if token_spans is None:
        return predictions, None, None

    # JSON 문자열 전체를 먼저 매칭해 값 내부의 키처럼 보이는 문자열을 제외합니다.
    string = r'"(?:[^"\\\x00-\x1f]|\\(?:["\\/bfnrt]|u[0-9a-fA-F]{4}))*"'
    number = r'-?(?:0|[1-9][0-9]*)(?:\.[0-9]+)?(?:[eE][+-]?[0-9]+)?'
    fields = list(re.finditer(rf'({string})\s*:\s*({string}|{number})|{string}', pred_text))
    fields = [match for match in fields if match.group(1) is not None]
    if len(fields) != 3 * len(data):
        return predictions, None, None
    byte_offsets = [0]
    for char in pred_text:
        byte_offsets.append(byte_offsets[-1] + len(char.encode("utf-8")))
    value_indices = set()
    for index, match in enumerate(fields):
        field = json.loads(match.group(1))
        start, end = match.span(2)
        if pred_text[start] == '"':
            start, end = start + 1, end - 1
        selected = {i for i, (left, right, _) in enumerate(token_spans)
                    if max(byte_offsets[start], left) < min(byte_offsets[end], right)}
        if selected:
            probabilities = [token_spans[i][2] for i in selected]
            predictions[index // 3][f"{field}_confidence"] = sum(probabilities) / len(probabilities)
            value_indices.update(selected)
    probabilities = [token_spans[i][2] for i in value_indices]
    return (predictions, sum(probabilities) / len(probabilities), min(probabilities)) if probabilities else (predictions, None, None)


def summarize_cases(cases):
    values = [case["value_min_confidence"] for case in cases if case["value_min_confidence"] is not None]
    averages = [case["value_avg_confidence"] for case in cases if case["value_avg_confidence"] is not None]
    correct = sum(case["is_exact_match"] for case in cases)
    return {
        "total": len(cases),
        "correct": correct,
        "accuracy": 100.0 * correct / len(cases) if cases else None,
        "value_min_confidence": min(values) if values else None,
        "value_avg_confidence": sum(averages) / len(averages) if averages else None,
        "confidence_count": len(values),
        "confidence_missing_count": len(cases) - len(values),
    }


## 6. 토크나이저와 4비트 모델


In [16]:
def load_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(
        runtime_path("adapter_path"), **HYPER_PARAMS["tokenizer"],
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_model():
    dtype = resolve_dtype(HYPER_PARAMS["precision"]["compute_dtype"])
    embedding_dtype = resolve_dtype(HYPER_PARAMS["precision"]["frozen_embedding_dtype"])
    model = AutoModelForCausalLM.from_pretrained(
        RUNTIME_PARAMS["model_name"], revision=RUNTIME_PARAMS["model_revision"],
        dtype=dtype,
        quantization_config=BitsAndBytesConfig(
            bnb_4bit_compute_dtype=dtype, **HYPER_PARAMS["quantization"],
        ),
        **HYPER_PARAMS["model_loading"],
    )
    for module in model.modules():
        if "norm" in type(module).__name__.lower():
            module.to(dtype=torch.float32)
    for module in (model.get_input_embeddings(), model.get_output_embeddings()):
        module.to(dtype=embedding_dtype)
    model = PeftModel.from_pretrained(model, runtime_path("adapter_path"), is_trainable=False)
    model.config.use_cache = True
    model.eval()
    return model


In [17]:
def prepare_evaluation(tokenizer):
    samples = read_jsonl(runtime_path("test_file"))
    prompts = [tokenizer.apply_chat_template(
        sample["messages"][:2], tokenize=False, add_generation_prompt=True,
        **HYPER_PARAMS["chat_template"],
    ) for sample in samples]
    paths = [runtime_path(key) for key in ("test_file", "split_meta_file", "run_config_file")]
    adapter = Path(runtime_path("adapter_path"))
    paths.extend(str(adapter / name) for name in
                 ("adapter_config.json", "adapter_model.safetensors", "tokenizer.json", "tokenizer_config.json"))
    return samples, prompts, {path: file_sha256(path) for path in paths}


## 7. 실행 정보와 평가


In [18]:
def collect_execution_metadata(model=None):
    from importlib.metadata import PackageNotFoundError, version

    packages = {}
    for name in ("torch", "transformers", "peft", "accelerate", "bitsandbytes"):
        try:
            packages[name] = version(name)
        except PackageNotFoundError:
            packages[name] = None
    return {
        "python": platform.python_version(), "python_executable": sys.executable,
        "platform": platform.platform(), "packages": packages,
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "cuda": torch.version.cuda,
        "base_model": RUNTIME_PARAMS["model_name"],
        "base_model_revision": getattr(getattr(model, "config", None), "_commit_hash", None)
            or RUNTIME_PARAMS["model_revision"],
        "compute_dtype": str(resolve_dtype(HYPER_PARAMS["precision"]["compute_dtype"])),
        "input_embedding_dtype": str(model.get_input_embeddings().weight.dtype) if model else None,
        "output_embedding_dtype": str(model.get_output_embeddings().weight.dtype) if model else None,
        "gpu_total_bytes": torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
        "peak_allocated_bytes": torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None,
        "peak_reserved_bytes": torch.cuda.max_memory_reserved() if torch.cuda.is_available() else None,
    }


In [19]:
def evaluate():
    report_path = Path(runtime_path("report_file"))
    if report_path.exists():
        raise FileExistsError(f"기존 평가 결과가 있습니다: {report_path}")
    settings = HYPER_PARAMS["evaluation"]
    tokenizer = load_tokenizer()
    samples, prompts, file_hashes = prepare_evaluation(tokenizer)
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    model = load_model()
    eos_ids = {tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")}
    eos_ids.discard(None)
    case_results = []
    print(f"평가 {len(samples)}건 / GPU: {torch.cuda.get_device_name(0)}")

    for start in tqdm(range(0, len(samples), settings["batch_size"]), desc="Evaluation"):
        batch = samples[start:start + settings["batch_size"]]
        inputs = tokenizer(prompts[start:start + len(batch)], return_tensors="pt", padding=True,
                           truncation=False, add_special_tokens=False).to(model.device)
        with torch.inference_mode(), torch.autocast(
                device_type="cuda", dtype=resolve_dtype(HYPER_PARAMS["precision"]["compute_dtype"])):
            outputs = model.generate(
                **inputs,
                max_new_tokens=settings["max_new_tokens"],
                do_sample=settings["do_sample"],
                temperature=None, top_p=None, top_k=None,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=sorted(eos_ids),
                return_dict_in_generate=True,
                output_scores=True,
            )
        generated = outputs.sequences[:, inputs["input_ids"].shape[1]:]
        for offset, sample in enumerate(batch):
            token_ids = generated[offset].tolist()
            end = next((i for i, token_id in enumerate(token_ids)
                        if token_id in eos_ids or token_id == tokenizer.pad_token_id), len(token_ids))
            terminated = end < len(token_ids) and token_ids[end] in eos_ids
            token_ids = token_ids[:end]
            pred_text = tokenizer.decode(token_ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)
            probabilities = [torch.softmax(outputs.scores[i][offset].float(), dim=-1)[token_id].item()
                             for i, token_id in enumerate(token_ids)]
            token_spans = build_token_spans(tokenizer, token_ids, probabilities, pred_text)
            prediction, avg_confidence, min_confidence = extract_values_with_confidence(pred_text, token_spans)
            ground_truth = sample["messages"][2]["content"]
            norm_pred = parse_and_normalize(pred_text)
            is_match = norm_pred is not None and Counter(norm_pred) == Counter(parse_and_normalize(ground_truth))
            metadata = sample["metadata"]
            case_results.append({
                "case_id": start + offset + 1,
                **{key: metadata[key] for key in ("site", "image_id", "source_index", "test_type")},
                "input_user": sample["messages"][1]["content"],
                "ground_truth_raw": ground_truth,
                "prediction_raw": pred_text,
                "prediction": prediction,
                "is_valid_json_schema": norm_pred is not None,
                "is_exact_match": is_match,
                "output_limit_reached": not terminated and end >= settings["max_new_tokens"],
                "value_avg_confidence": avg_confidence,
                "value_min_confidence": min_confidence,
            })
        # 다음 배치를 생성하기 전에 어휘 점수를 해제합니다.
        del outputs, inputs, generated

    overall = summarize_cases(case_results)
    by_type = {name: summarize_cases([case for case in case_results if case["test_type"] == name])
               for name in sorted({case["test_type"] for case in case_results})}
    report = {
        "model": RUNTIME_PARAMS["model_name"],
        "adapter": runtime_path("adapter_path"),
        "device": torch.cuda.get_device_name(0),
        "dtype": str(model.dtype),
        "quantized": True, "adapter_merged": False,
        "duration_seconds": time.perf_counter() - started,
        "test_file": runtime_path("test_file"),
        "runtime_params": deepcopy(RUNTIME_PARAMS),
        "hyper_params": deepcopy(HYPER_PARAMS),
        "batch_size": settings["batch_size"],
        "scoring_version": "strict_schema_multiset_v1",
        "confidence_definition": "Generated tokens overlapping workType/spec/quantity value bytes; unique tokens per case. Group min includes correct and incorrect cases with known confidence; null cases are excluded and counted. Avg is the mean of known case averages.",
        "file_hashes": file_hashes,
        "execution": collect_execution_metadata(model),
        **overall,
        "correct_value_avg_confidence": summarize_cases([case for case in case_results if case["is_exact_match"]])["value_avg_confidence"],
        "incorrect_value_avg_confidence": summarize_cases([case for case in case_results if not case["is_exact_match"]])["value_avg_confidence"],
        "by_test_type": by_type,
        "cases": case_results,
    }
    write_json(report_path, report)
    print(f"완전 일치율: {overall['accuracy']:.2f}% ({overall['correct']}/{overall['total']})")
    print(f"평가 결과 저장: {report_path}")
    return report


## 8. 설정 확인 후 실행


In [20]:
print(json.dumps(RUNTIME_PARAMS, ensure_ascii=False, indent=2))
print(json.dumps(HYPER_PARAMS, ensure_ascii=False, indent=2))


{
  "prefix_path": "./data/",
  "model_name": "Qwen/Qwen3-8B",
  "model_revision": "b968826d9c46dd6066d109eabc6255188de91218",
  "adapter_path": "models",
  "test_file": "test.jsonl",
  "split_meta_file": "split_meta.json",
  "run_config_file": "result/run_config.json",
  "report_file": "result/evaluation.json",
  "device": "cuda"
}
{
  "chat_template": {
    "enable_thinking": false
  },
  "quantization": {
    "load_in_4bit": true,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true
  },
  "precision": {
    "compute_dtype": "auto",
    "frozen_embedding_dtype": "auto"
  },
  "model_loading": {
    "device_map": {
      "": 0
    },
    "attn_implementation": "sdpa",
    "low_cpu_mem_usage": true
  },
  "tokenizer": {
    "padding_side": "left"
  },
  "evaluation": {
    "batch_size": 1,
    "max_new_tokens": 512,
    "do_sample": false
  }
}


In [21]:
report = evaluate()


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

평가 555건 / GPU: NVIDIA GeForce RTX 4070


Evaluation: 100%|██████████| 555/555 [1:09:10<00:00,  7.48s/it]

완전 일치율: 87.93% (488/555)
평가 결과 저장: data\result\evaluation.json
